In [36]:
import pandas as pd
import numpy as np
import json
import pickle
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')

# CONFIG
OUTPUT_DIR = '../../model_outputs'
DATASET_PATH = '../../dataset/dataset_sapi_lstm.csv'
WINDOW = 7  # Data 14 sesi sebelumnya untuk prediksi 1 sesi depan
TARGET = 'jumlah_susu'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Folder output siap: {OUTPUT_DIR}")

✅ Folder output siap: ../../model_outputs


In [37]:
df = pd.read_csv(DATASET_PATH)

# 1. Filter: Hapus data di mana jumlah_susu adalah 0
df = df[df[TARGET] > 0]

# 2. Hapus kolom yang tidak diperlukan untuk pelatihan
# Kita simpan 'sapi_id' sebentar untuk proses windowing, lalu hapus nanti
DROP_COLS = ['nama_sapi', 'pemerahan_id', 'tgl_lahir', 'pemerah']
df = df.drop(columns=DROP_COLS)

# 3. Parse Tanggal
df['tgl_pemerahan'] = pd.to_datetime(df['tgl_pemerahan'], errors='coerce')
df = df.sort_values(['sapi_id', 'tgl_pemerahan']).reset_index(drop=True)

print(f"Shape setelah cleaning: {df.shape}")
display(df.head(3))

Shape setelah cleaning: (95, 8)


,sapi_id,jenis_sapi,tgl_pemerahan,jumlah_susu,volume_pakan,jenis_pakan,kondisi_sapi,status_reproduksi
0,1azOwdcfRGYY5QyJdQsa,perah peranakan FH,2025-11-27 09:03:29,14,52,Campuran,Sehat,Laktasi
1,1azOwdcfRGYY5QyJdQsa,perah peranakan FH,2025-11-28 08:30:08,11,52,Campuran,Sehat,Laktasi
2,1azOwdcfRGYY5QyJdQsa,perah peranakan FH,2025-11-29 12:03:21,14,53,Campuran,Sehat,Laktasi


In [38]:
# Ekstrak fitur waktu
df['bulan'] = df['tgl_pemerahan'].dt.month
df['hari_minggu'] = df['tgl_pemerahan'].dt.dayofweek
df['jam'] = df['tgl_pemerahan'].dt.hour

# Label Encoding untuk kategori sisanya
CATEGORICAL_COLS = ['jenis_sapi', 'jenis_pakan', 'kondisi_sapi', 'status_reproduksi']
encoders = {}

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    # Pastikan data string dan seragam
    df[col] = df[col].astype(str).str.lower().str.strip()
    df[col + '_enc'] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"Encoded {col}: {list(le.classes_)}")

# Hapus kolom asli setelah di-encode
df_final = df.drop(columns=CATEGORICAL_COLS + ['tgl_pemerahan'])
print(f"\nKolom Akhir: {list(df_final.columns)}")

Encoded jenis_sapi: ['perah jenis fh', 'perah perabakan fh', 'perah peranakan fh', 'sapi perah jenis fh']
Encoded jenis_pakan: ['campuran', 'konsentrat', 'rumput']
Encoded kondisi_sapi: ['sehat']
Encoded status_reproduksi: ['hamil', 'laktasi']

Kolom Akhir: ['sapi_id', 'jumlah_susu', 'volume_pakan', 'bulan', 'hari_minggu', 'jam', 'jenis_sapi_enc', 'jenis_pakan_enc', 'kondisi_sapi_enc', 'status_reproduksi_enc']


In [39]:
FEATURE_COLS = [c for c in df_final.columns if c not in [TARGET, 'sapi_id']]
n_features = len(FEATURE_COLS)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

all_X, all_y = [], []

for cow_id, group in df_final.groupby('sapi_id'):
    group = group.reset_index(drop=True)
    if len(group) <= WINDOW:
        continue

    features = group[FEATURE_COLS].values.astype(float)
    target = group[TARGET].values.astype(float).reshape(-1, 1)

    for i in range(WINDOW, len(group)):
        all_X.append(features[i-WINDOW:i])
        all_y.append(target[i])

all_X = np.array(all_X, dtype=np.float32)
all_y = np.array(all_y, dtype=np.float32)

# Scaling
all_X_scaled = scaler_X.fit_transform(all_X.reshape(-1, n_features)).reshape(-1, WINDOW, n_features)
all_y_scaled = scaler_y.fit_transform(all_y)

print(f"Total Sequence: {len(all_X_scaled)}")

Total Sequence: 31


In [40]:
X_train, X_test, y_train, y_test = train_test_split(all_X_scaled, all_y_scaled, test_size=0.2, random_state=42)

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, n_features)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.1, verbose=1)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 483ms/step - loss: 0.5108 - mae: 0.6411 - val_loss: 0.5262 - val_mae: 0.6998
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.4226 - mae: 0.5700 - val_loss: 0.4504 - val_mae: 0.6411
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.3684 - mae: 0.5421 - val_loss: 0.3862 - val_mae: 0.5876
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.2965 - mae: 0.4819 - val_loss: 0.3200 - val_mae: 0.5266
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.2595 - mae: 0.4500 - val_loss: 0.2492 - val_mae: 0.4530
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 0.1972 - mae: 0.4061 - val_loss: 0.1779 - val_mae: 0.3653
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.1491 - mae: 0.3541 - val_loss: 0.1147 - val_mae: 0.2780
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.1188 - mae: 0.3029 - val_loss: 0.0704 - val_mae: 0.2415
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 0.1250 - mae:

In [ ]:
import tf2onnx

# 1. Evaluasi
y_pred = scaler_y.inverse_transform(model.predict(X_test))
y_true = scaler_y.inverse_transform(y_test)
print(f"MAE: {mean_absolute_error(y_true, y_pred):.2f} Liter")

# 2. Save Artifacts
with open(f'{OUTPUT_DIR}/scaler_X.pkl', 'wb') as f: pickle.dump(scaler_X, f)
with open(f'{OUTPUT_DIR}/scaler_y.pkl', 'wb') as f: pickle.dump(scaler_y, f)
with open(f'{OUTPUT_DIR}/encoders.pkl', 'wb') as f: pickle.dump(encoders, f)

# JSON untuk Android
params = {
    "X_min": scaler_X.data_min_.tolist(), 
    "X_max": scaler_X.data_max_.tolist(),
    "y_min": float(scaler_y.data_min_[0]), 
    "y_max": float(scaler_y.data_max_[0]),
    "window": WINDOW, 
    "n_features": n_features,
    "feature_names": FEATURE_COLS,
    "categorical_encodings": {
        col: {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))} 
        for col, le in encoders.items()
    }
}
with open(f'{OUTPUT_DIR}/scaler_params.json', 'w') as f: json.dump(params, f, indent=2)

# 3. Export Model
functional_model = tf.keras.Model(inputs=model.inputs, outputs=model.outputs)
functional_model.save(f'{OUTPUT_DIR}/milk_lstm.keras')

# TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(functional_model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
with open(f'{OUTPUT_DIR}/milk_lstm.tflite', 'wb') as f: f.write(converter.convert())

# ONNX
spec = (tf.TensorSpec((None, WINDOW, n_features), tf.float32, name="input"),)
onnx_model, _ = tf2onnx.convert.from_keras(functional_model, input_signature=spec, opset=13)
with open(f'{OUTPUT_DIR}/milk_lstm.onnx', 'wb') as f: f.write(onnx_model.SerializeToString())

print(f"✅ Semua file berhasil disimpan di folder: {OUTPUT_DIR}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step
MAE: 2.73 Liter
INFO:tensorflow:Assets written to: C:\Users\nawfal\AppData\Local\Temp\tmpidd93d4e\assets


INFO:tensorflow:Assets written to: C:\Users\nawfal\AppData\Local\Temp\tmpidd93d4e\assets


Saved artifact at 'C:\Users\nawfal\AppData\Local\Temp\tmpidd93d4e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7, 8), dtype=tf.float32, name='keras_tensor_21')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2146470553984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470089600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470092416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470085024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470092768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470094880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470094704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470091536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470089248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2146470088016: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Semua fi

In [35]:
# ──────────────────────────────────────────────
# 10. EXPORT → ONNX (Robust Version)
# ──────────────────────────────────────────────
import tf2onnx
import onnx
import os

OUTPUT_DIR = 'model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Wrap ke Functional Model untuk menstabilkan nama tensor
functional_model = tf.keras.Model(inputs=model.inputs, outputs=model.outputs)

# Definisikan signature input: [Batch, Window, Features]
input_sig = [tf.TensorSpec(shape=[None, WINDOW, n_features], dtype=tf.float32, name='input')]

# Konversi
onnx_model, _ = tf2onnx.convert.from_keras(
    functional_model, 
    input_signature=input_sig, 
    opset=13
)

# Simpan file
onnx_path = os.path.join(OUTPUT_DIR, 'milk_lstm.onnx')
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f"✅ ONNX berhasil disimpan di: {onnx_path}")

✅ ONNX berhasil disimpan di: model_outputs\milk_lstm.onnx
